# Week 4 Day 4: Pandas ETL to Snowflake

Build a full pipeline: extract, profile, rename, clean, validate, engineer, analyze, publish, and reconcile.

This is an AI-Free Zone activity. Write and explain your own code. Never place a password, token, AWS key, or populated connection value in this notebook.

## How this notebook teaches

- **Run as written:** setup or unfamiliar mechanics are provided.
- **Complete a focused TODO:** some code is provided and you add the missing pandas operation.
- **Write the entire cell:** later work gives an exact contract and expects you to assemble the solution.

The challenge should come from applying skills, not guessing an unstated formula or connection pattern.


## 1. Environment and imports

**Run as written.** The notebook uses `ConfigParser` to read local Snowflake context from `snow.cfg`. The connection file is not imported into the notebook and is ignored by the repository-root `.gitignore`.


In [3]:
from configparser import ConfigParser
from pathlib import Path

import numpy as np
import pandas as pd
import snowflake.connector
from snowflake.connector.pandas_tools import write_pandas

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")


## 2. Extract and profile

**Run as written.** Read the approved public CSV from S3. The expected source shape is 463,152 rows and 27 columns.


In [4]:
ACCIDENTS_S3_URI = (
    "s3://techcatalyst-de-2026/raw/accidents/"
    "accidents_2017_to_2023_english.csv"
)

raw_accidents = pd.read_csv(
    ACCIDENTS_S3_URI,
    storage_options={"anon": True},
)

print({"rows": len(raw_accidents), "columns": len(raw_accidents.columns)})
assert raw_accidents.shape == (463_152, 27), (
    "The source shape changed. Stop and confirm the approved file."
)
raw_accidents.head()


{'rows': 463152, 'columns': 27}


,inverse_data,week_day,hour,state,road_id,km,city,cause_of_accident,type_of_accident,victims_condition,weather_timestamp,road_direction,wheather_condition,road_type,road_delineation,people,deaths,slightly_injured,severely_injured,uninjured,ignored,total_injured,vehicles_involved,latitude,longitude,regional,police_station
0,2017-01-01,sunday,01:45:00,RS,116.00,"34,9",VACARIA,Mechanical loss/defect of vehicle,Rear-end collision,With injured victims,Night,Decreasing,Clear sky,Simple,Straight,6,0,4,0,2,0,4,2,-28.51,-50.94,SPRF-RS,DEL05-RS
1,2017-01-01,sunday,01:00:00,PR,376.00,636,TIJUCAS DO SUL,Incompatible velocity,Run-off-road,With dead victims,Night,Increasing,Drizzle,Double,Curve,2,1,0,0,1,0,0,2,-25.75,-49.13,SPRF-PR,DEL01-PR
2,2017-01-01,sunday,04:40:00,BA,101.00,65,ENTRE RIOS,Driver was sleeping,Head-on collision,With dead victims,Sunrise,Decreasing,Cloudy,Simple,Curve,5,1,1,1,2,0,2,2,-11.96,-38.10,SPRF-BA,DEL01-BA
3,2017-01-01,sunday,06:30:00,PA,316.00,"72,5",CASTANHAL,Driver's lack of attention to conveyance,Side impact collision,With dead victims,Sunrise,Decreasing,Clear sky,Simple,Straight,4,1,0,0,3,0,0,3,-1.29,-47.83,SPRF-PA,DEL01-PA
4,2017-01-01,sunday,09:00:00,GO,20.00,"220,5",POSSE,Road's defect,Collision with fixed object,With injured victims,Day,Decreasing,Clear sky,Simple,Temporary Detour,3,0,2,1,0,0,3,1,-14.14,-46.32,SPRF-DF,DEL02-DF


### Your work: profile the source

Write this entire cell.

1. Display the list of column names.
2. Display a two-column table containing each source column and dtype.
3. Display the first five rows.
4. Add two comments identifying fields that need type correction.

**Expected observations:** `inverse_data` is text but represents a date. `km` contains comma decimal notation and must become numeric.


In [ ]:
# TODO: Write the full profiling cell described above.


['inverse_data', 'week_day', 'hour', 'state', 'road_id', 'km', 'city', 'cause_of_accident', 'type_of_accident', 'victims_condition', 'weather_timestamp', 'road_direction', 'wheather_condition', 'road_type', 'road_delineation', 'people', 'deaths', 'slightly_injured', 'severely_injured', 'uninjured', 'ignored', 'total_injured', 'vehicles_involved', 'latitude', 'longitude', 'regional', 'police_station']


,source_dtype
inverse_data,str
week_day,str
hour,str
state,str
road_id,float64
km,str
city,str
cause_of_accident,str
type_of_accident,str
victims_condition,str


,inverse_data,week_day,hour,state,road_id,km,city,cause_of_accident,type_of_accident,victims_condition,weather_timestamp,road_direction,wheather_condition,road_type,road_delineation,people,deaths,slightly_injured,severely_injured,uninjured,ignored,total_injured,vehicles_involved,latitude,longitude,regional,police_station
0,2017-01-01,sunday,01:45:00,RS,116.00,"34,9",VACARIA,Mechanical loss/defect of vehicle,Rear-end collision,With injured victims,Night,Decreasing,Clear sky,Simple,Straight,6,0,4,0,2,0,4,2,-28.51,-50.94,SPRF-RS,DEL05-RS
1,2017-01-01,sunday,01:00:00,PR,376.00,636,TIJUCAS DO SUL,Incompatible velocity,Run-off-road,With dead victims,Night,Increasing,Drizzle,Double,Curve,2,1,0,0,1,0,0,2,-25.75,-49.13,SPRF-PR,DEL01-PR
2,2017-01-01,sunday,04:40:00,BA,101.00,65,ENTRE RIOS,Driver was sleeping,Head-on collision,With dead victims,Sunrise,Decreasing,Cloudy,Simple,Curve,5,1,1,1,2,0,2,2,-11.96,-38.10,SPRF-BA,DEL01-BA
3,2017-01-01,sunday,06:30:00,PA,316.00,"72,5",CASTANHAL,Driver's lack of attention to conveyance,Side impact collision,With dead victims,Sunrise,Decreasing,Clear sky,Simple,Straight,4,1,0,0,3,0,0,3,-1.29,-47.83,SPRF-PA,DEL01-PA
4,2017-01-01,sunday,09:00:00,GO,20.00,"220,5",POSSE,Road's defect,Collision with fixed object,With injured victims,Day,Decreasing,Clear sky,Simple,Temporary Detour,3,0,2,1,0,0,3,1,-14.14,-46.32,SPRF-DF,DEL02-DF


## 3. Build the rename mapping

The source uses several unclear or misspelled names. Build a Python dictionary named `COLUMN_MAP` with these exact source-to-target pairs:

| Source column | Target column |
|---|---|
| `inverse_data` | `accident_date` |
| `week_day` | `day_of_week` |
| `hour` | `accident_time` |
| `state` | `state_code` |
| `road_id` | `highway_number` |
| `km` | `kilometer_marker` |
| `city` | `city_name` |
| `cause_of_accident` | `accident_cause` |
| `type_of_accident` | `accident_type` |
| `victims_condition` | `casualty_status` |
| `weather_timestamp` | `daylight_condition` |
| `road_direction` | `traffic_direction` |
| `wheather_condition` | `weather_condition` |
| `road_type` | `road_classification` |
| `road_delineation` | `road_geometry` |
| `people` | `total_people_involved` |
| `deaths` | `fatalities` |
| `slightly_injured` | `minor_injuries` |
| `severely_injured` | `serious_injuries` |
| `uninjured` | `uninjured_count` |
| `ignored` | `unknown_status` |
| `total_injured` | `total_injuries` |
| `vehicles_involved` | `vehicle_count` |
| `latitude` | `accident_latitude` |
| `longitude` | `accident_longitude` |
| `regional` | `regional_office` |
| `police_station` | `reporting_station` |

Two entries are started for you. Add the other 25. Do not copy a completed dictionary from the solution folder.


In [ ]:
COLUMN_MAP = {
    "inverse_data": "accident_date",
    "week_day": "day_of_week",
    # TODO: Add the remaining 25 source-to-target pairs from the table above.
}


### Apply and verify the mapping

Complete the focused TODOs:

1. Use `raw_accidents.rename(columns=COLUMN_MAP)` and copy the result into `renamed_accidents`.
2. Calculate which required columns are missing.
3. Assert that the missing set is empty.

If the assertion fails, compare your dictionary with the instruction table. Do not continue with a partially renamed schema.


In [ ]:
REQUIRED_ANALYSIS_COLUMNS = {
    "accident_date",
    "day_of_week",
    "accident_time",
    "state_code",
    "accident_cause",
    "weather_condition",
    "road_classification",
    "fatalities",
    "minor_injuries",
    "serious_injuries",
    "total_injuries",
    "vehicle_count",
}

# TODO: Rename the source without changing raw_accidents.
# renamed_accidents =

# TODO: Find required names that are missing from renamed_accidents.
# missing_required =

# TODO: Assert that missing_required is empty.

print(f"Mapped {len(COLUMN_MAP)} source columns.")


Mapped 27 source columns.


,accident_date,day_of_week,accident_time,state_code,highway_number,kilometer_marker,city_name,accident_cause,accident_type,casualty_status,daylight_condition,traffic_direction,weather_condition,road_classification,road_geometry,total_people_involved,fatalities,minor_injuries,serious_injuries,uninjured_count,unknown_status,total_injuries,vehicle_count,accident_latitude,accident_longitude,regional_office,reporting_station
0,2017-01-01,sunday,01:45:00,RS,116.00,"34,9",VACARIA,Mechanical loss/defect of vehicle,Rear-end collision,With injured victims,Night,Decreasing,Clear sky,Simple,Straight,6,0,4,0,2,0,4,2,-28.51,-50.94,SPRF-RS,DEL05-RS
1,2017-01-01,sunday,01:00:00,PR,376.00,636,TIJUCAS DO SUL,Incompatible velocity,Run-off-road,With dead victims,Night,Increasing,Drizzle,Double,Curve,2,1,0,0,1,0,0,2,-25.75,-49.13,SPRF-PR,DEL01-PR
2,2017-01-01,sunday,04:40:00,BA,101.00,65,ENTRE RIOS,Driver was sleeping,Head-on collision,With dead victims,Sunrise,Decreasing,Cloudy,Simple,Curve,5,1,1,1,2,0,2,2,-11.96,-38.10,SPRF-BA,DEL01-BA
3,2017-01-01,sunday,06:30:00,PA,316.00,"72,5",CASTANHAL,Driver's lack of attention to conveyance,Side impact collision,With dead victims,Sunrise,Decreasing,Clear sky,Simple,Straight,4,1,0,0,3,0,0,3,-1.29,-47.83,SPRF-PA,DEL01-PA
4,2017-01-01,sunday,09:00:00,GO,20.00,"220,5",POSSE,Road's defect,Collision with fixed object,With injured victims,Day,Decreasing,Clear sky,Simple,Temporary Detour,3,0,2,1,0,0,3,1,-14.14,-46.32,SPRF-DF,DEL02-DF


### Rename checkpoint

Your mapping must contain 27 entries. `renamed_accidents` should still have 463,152 rows and 27 columns. Its first fields should be `accident_date`, `day_of_week`, `accident_time`, and `state_code`.


## 4. Clean types and values

This function is intentionally mixed:

- The date conversion is completed as an example.
- The time parser is provided, and you create `accident_hour`.
- The comma-decimal requirement and pandas methods are stated.
- The loops are provided, and you complete the conversion expressions.
- The required-row list is provided so you do not guess which missing values justify dropping a row.

### Required behavior

1. Convert `accident_date` with `pd.to_datetime(..., errors="coerce")`.
2. Parse `accident_time` with format `%H:%M:%S` and store the hour as nullable `Int64`.
3. Replace commas with periods in `kilometer_marker` before `pd.to_numeric`.
4. Trim and lowercase the listed text fields.
5. Trim and uppercase `state_code`.
6. Convert the listed measures with `pd.to_numeric(..., errors="coerce")`.
7. Drop rows only when a field required by the four analyses is unusable.


In [8]:
TEXT_COLUMNS = [
    "day_of_week",
    "city_name",
    "accident_cause",
    "accident_type",
    "casualty_status",
    "daylight_condition",
    "traffic_direction",
    "weather_condition",
    "road_classification",
    "road_geometry",
]

NUMERIC_COLUMNS = [
    "total_people_involved",
    "fatalities",
    "minor_injuries",
    "serious_injuries",
    "uninjured_count",
    "unknown_status",
    "total_injuries",
    "vehicle_count",
]

REQUIRED_AFTER_CLEANING = [
    "accident_date",
    "accident_hour",
    "day_of_week",
    "state_code",
    "accident_cause",
    "weather_condition",
    "road_classification",
    "fatalities",
    "minor_injuries",
    "serious_injuries",
    "total_injuries",
    "vehicle_count",
]


In [ ]:
def clean_accidents(frame):
    cleaned = frame.copy()

    # Completed example: text date to datetime.
    cleaned["accident_date"] = pd.to_datetime(
        cleaned["accident_date"],
        errors="coerce",
    )

    # The parser is provided.
    parsed_time = pd.to_datetime(
        cleaned["accident_time"].astype("string"),
        format="%H:%M:%S",
        errors="coerce",
    )

    # TODO 1: Extract parsed_time.dt.hour and convert it to nullable Int64.

    # TODO 2: Convert kilometer_marker.
    # Hint: astype("string"), str.replace(",", ".", regex=False),
    # then pd.to_numeric(..., errors="coerce").

    for column in TEXT_COLUMNS:
        # TODO 3: Convert to string, trim whitespace, and lowercase.
        pass

    # TODO 4: Standardize state_code as trimmed uppercase text.

    for column in NUMERIC_COLUMNS:
        # TODO 5: Convert each measure with errors="coerce".
        pass

    rows_before = len(cleaned)

    # TODO 6: Drop rows only when a REQUIRED_AFTER_CLEANING field is missing.

    print({
        "rows_before": rows_before,
        "rows_after": len(cleaned),
        "rows_removed": rows_before - len(cleaned),
    })
    return cleaned

In [10]:
cleaned_accidents = clean_accidents(renamed_accidents)

display(cleaned_accidents.head())
display(cleaned_accidents.dtypes.rename("cleaned_dtype").to_frame())

assert len(cleaned_accidents) == 463_152
assert str(cleaned_accidents["accident_hour"].dtype) == "Int64"
assert pd.api.types.is_datetime64_any_dtype(cleaned_accidents["accident_date"])


{'rows_before': 463152, 'rows_after': 463152, 'rows_removed': 0}


,accident_date,day_of_week,accident_time,state_code,highway_number,kilometer_marker,city_name,accident_cause,accident_type,casualty_status,daylight_condition,traffic_direction,weather_condition,road_classification,road_geometry,total_people_involved,fatalities,minor_injuries,serious_injuries,uninjured_count,unknown_status,total_injuries,vehicle_count,accident_latitude,accident_longitude,regional_office,reporting_station,accident_hour
0,2017-01-01,sunday,01:45:00,RS,116.00,34.90,vacaria,mechanical loss/defect of vehicle,rear-end collision,with injured victims,night,decreasing,clear sky,simple,straight,6,0,4,0,2,0,4,2,-28.51,-50.94,SPRF-RS,DEL05-RS,1
1,2017-01-01,sunday,01:00:00,PR,376.00,636.00,tijucas do sul,incompatible velocity,run-off-road,with dead victims,night,increasing,drizzle,double,curve,2,1,0,0,1,0,0,2,-25.75,-49.13,SPRF-PR,DEL01-PR,1
2,2017-01-01,sunday,04:40:00,BA,101.00,65.00,entre rios,driver was sleeping,head-on collision,with dead victims,sunrise,decreasing,cloudy,simple,curve,5,1,1,1,2,0,2,2,-11.96,-38.10,SPRF-BA,DEL01-BA,4
3,2017-01-01,sunday,06:30:00,PA,316.00,72.50,castanhal,driver's lack of attention to conveyance,side impact collision,with dead victims,sunrise,decreasing,clear sky,simple,straight,4,1,0,0,3,0,0,3,-1.29,-47.83,SPRF-PA,DEL01-PA,6
4,2017-01-01,sunday,09:00:00,GO,20.00,220.50,posse,road's defect,collision with fixed object,with injured victims,day,decreasing,clear sky,simple,temporary detour,3,0,2,1,0,0,3,1,-14.14,-46.32,SPRF-DF,DEL02-DF,9


,cleaned_dtype
accident_date,datetime64[us]
day_of_week,string
accident_time,str
state_code,string
highway_number,float64
kilometer_marker,Float64
city_name,string
accident_cause,string
accident_type,string
casualty_status,string


### Cleaning checkpoint

Expected evidence:

- Rows before: 463,152
- Rows after: 463,152
- Rows removed: 0
- `accident_date` is datetime
- `accident_hour` is nullable `Int64`
- `kilometer_marker` is numeric
- Text categories are trimmed and lowercase
- `state_code` is uppercase

Rows with missing `reporting_station`, `highway_number`, `kilometer_marker`, or `regional_office` remain because those fields are not required by the four analyses.


## 5. Data-quality evidence

### 5A. Missing-value profile

Write a table with `column_name`, `missing_count`, and `missing_percent`. Keep only columns with at least one missing value and sort from most missing to least.


In [ ]:
missing_counts = cleaned_accidents.isna().sum()

# TODO: Starting with missing_counts, build missing_profile with:
# column_name, missing_count, and missing_percent.
# Keep missing_count > 0 and sort descending.

# display(missing_profile)


,column_name,missing_count,missing_percent
0,reporting_station,1310,0.28
1,highway_number,990,0.21
2,kilometer_marker,990,0.21
3,regional_office,10,0.00


**Expected missing counts:** reporting station 1,310; highway number 990; kilometer marker 990; regional office 10.


### 5B. Injury-integrity check

The source field `total_injuries` represents minor plus serious injuries. It does not include fatalities.

Create:

`calculated_total_injuries = minor_injuries + serious_injuries`

Then create `injury_total_match` and report the mismatch count and percentage.


In [ ]:
# TODO 1: Create calculated_total_injuries.
# TODO 2: Create the Boolean injury_total_match column.
# TODO 3: Calculate injury_mismatches and mismatch_percent.
# TODO 4: Print both values and assert that mismatches equal zero.


{'injury_mismatches': 0, 'mismatch_percent': 0.0}


### 5C. IQR outlier report

You are not expected to invent the IQR method from memory. Read the method, run the provided implementation, and interpret the evidence.

For one numerical column:

1. `Q1` is the 25th percentile.
2. `Q3` is the 75th percentile.
3. `IQR = Q3 - Q1`.
4. The lower boundary is `Q1 - 1.5 * IQR`.
5. The upper boundary is `Q3 + 1.5 * IQR`.
6. Values below the lower boundary or above the upper boundary are flagged for review.

An IQR flag is not proof that a record is wrong. For example, any fatality is statistically unusual when the 75th percentile is zero, but it may still be valid. Report these rows. Do not automatically delete them.


In [13]:
OUTLIER_COLUMNS = [
    "total_people_involved",
    "fatalities",
    "minor_injuries",
    "serious_injuries",
    "total_injuries",
    "vehicle_count",
]


def build_iqr_report(frame, columns):
    records = []

    for column in columns:
        series = frame[column].dropna()
        q1 = series.quantile(0.25)
        q3 = series.quantile(0.75)
        iqr = q3 - q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr

        outlier_mask = (
            (series < lower_bound)
            | (series > upper_bound)
        )

        records.append({
            "column_name": column,
            "q1": q1,
            "q3": q3,
            "lower_bound": lower_bound,
            "upper_bound": upper_bound,
            "outlier_count": int(outlier_mask.sum()),
            "outlier_percent": round(outlier_mask.mean() * 100, 2),
            "max_value": series.max(),
        })

    return pd.DataFrame(records)


outlier_report = build_iqr_report(
    cleaned_accidents,
    OUTLIER_COLUMNS,
)
display(outlier_report)

,column_name,q1,q3,lower_bound,upper_bound,outlier_count,outlier_percent,max_value
0,total_people_involved,1.00,3.00,-2.00,6.00,8587,1.85,80
1,fatalities,0.00,0.00,0.00,0.00,31344,6.77,21
2,minor_injuries,0.00,1.00,-1.50,2.50,20283,4.38,61
3,serious_injuries,0.00,0.00,0.00,0.00,96751,20.89,31
4,total_injuries,0.00,1.00,-1.50,2.50,30484,6.58,66
5,vehicle_count,1.00,2.00,-0.50,3.50,8574,1.85,23


### IQR checkpoint and interpretation

Expected outlier counts:

| Column | Outlier count | Maximum |
|---|---:|---:|
| total_people_involved | 8,587 | 80 |
| fatalities | 31,344 | 21 |
| minor_injuries | 20,283 | 61 |
| serious_injuries | 96,751 | 31 |
| total_injuries | 30,484 | 66 |
| vehicle_count | 8,574 | 23 |

Add a Markdown cell below and answer:

1. Why does `fatalities` have an upper IQR boundary of zero?
2. Why should that not cause every fatal accident to be deleted?
3. Which maximum value would you review first, and what additional evidence would you request?


## 6. Feature engineering

Create:

- `year`, `month`, `day`, and `quarter` from `accident_date`
- `time_of_day_category`: morning 05:00 through 11:59, afternoon 12:00 through 16:59, evening 17:00 through 20:59, night otherwise
- `weekend_flag`
- `fatal_accident_flag`
- `multi_vehicle_flag`
- `accident_severity_score = fatalities * 10 + serious_injuries * 3 + minor_injuries`

The `np.select` conditions and labels are provided. Complete the focused assignments.


In [ ]:
def engineer_features(frame):
    featured = frame.copy()

    # Completed example:
    featured["year"] = featured["accident_date"].dt.year

    # TODO 1: Create month, day, and quarter.

    time_conditions = [
        featured["accident_hour"].between(5, 11),
        featured["accident_hour"].between(12, 16),
        featured["accident_hour"].between(17, 20),
    ]
    time_labels = ["morning", "afternoon", "evening"]

    # TODO 2: Use np.select with time_conditions, time_labels,
    # and default="night".

    # TODO 3: Create weekend_flag.

    # TODO 4: Create fatal_accident_flag and multi_vehicle_flag.

    # TODO 5: Create accident_severity_score from the stated formula.

    return featured

In [15]:
featured_accidents = engineer_features(cleaned_accidents)

FEATURE_COLUMNS = [
    "year",
    "month",
    "day",
    "quarter",
    "time_of_day_category",
    "weekend_flag",
    "fatal_accident_flag",
    "multi_vehicle_flag",
    "accident_severity_score",
]

display(featured_accidents[FEATURE_COLUMNS].head())
assert set(FEATURE_COLUMNS).issubset(featured_accidents.columns)


,year,month,day,quarter,time_of_day_category,weekend_flag,fatal_accident_flag,multi_vehicle_flag,accident_severity_score
0,2017,1,1,1,night,True,False,True,4
1,2017,1,1,1,night,True,True,True,10
2,2017,1,1,1,night,True,True,True,14
3,2017,1,1,1,morning,True,True,True,10
4,2017,1,1,1,morning,True,False,False,5


### Feature checkpoint

The first row should represent January 1, 2017, during the night, on a weekend. Its severity score is 4. The first five rows should have severity scores 4, 10, 14, 10, and 5.


## 7. Create four business analysis DataFrames

Each analysis specifies its grain, exact output columns, sorting rule, and expected row count. Counts remain beside averages so a rare category is not mistaken for a reliable pattern.


### Analysis 1: Severity by weather and road classification

**Grain:** one row per weather and road-classification combination.

**Steps:**

1. Group by `weather_condition` and `road_classification`.
2. Aggregate `accident_count` with size.
3. Aggregate `average_severity_score` with mean.
4. Reset the index.
5. Round the average to two decimals.
6. Sort by average severity descending, then accident count descending.

**Required columns:** `weather_condition`, `road_classification`, `accident_count`, `average_severity_score`.  
**Expected rows:** 28.


In [18]:
# The group keys are provided. Complete the aggregation and finishing steps.
severity_groups = featured_accidents.groupby(
    ["weather_condition", "road_classification"],
    dropna=False,
)

# TODO: Build severity_analysis from severity_groups.
# Required named aggregations:
# accident_count=("accident_date", "size")
# average_severity_score=("accident_severity_score", "mean")

display(severity_analysis.head())
assert len(severity_analysis) == 28


,weather_condition,road_classification,accident_count,average_severity_score
11,fog,simple,2482,3.51
16,ignored,simple,3816,3.28
27,windy,simple,425,3.04
19,rainy,simple,25996,2.98
2,clear sky,simple,139712,2.93


**Checkpoint:** the first row should be fog plus simple road classification, with 2,482 accidents and an average severity score of 3.51.


### Analysis 2: Fatal accidents by time and weekend status

**Grain:** one row per time-of-day and weekend combination.

The filter is provided. Complete the grouping, count, reset, and descending sort.

**Required columns:** `time_of_day_category`, `weekend_flag`, `number_of_fatal_accidents`.  
**Expected rows:** 8.


In [20]:
fatal_accidents = featured_accidents.loc[
    featured_accidents["fatal_accident_flag"]
].copy()

# TODO: Group by time_of_day_category and weekend_flag.
# Use .size(), rename the count, reset the index, and sort descending.
# Save the result as fatal_temporal_patterns.

display(fatal_temporal_patterns)
assert len(fatal_temporal_patterns) == 8


,time_of_day_category,weekend_flag,number_of_fatal_accidents
2,evening,False,5447
6,night,False,5111
4,morning,False,4822
7,night,True,4165
0,afternoon,False,3684
3,evening,True,3592
5,morning,True,2556
1,afternoon,True,1967


**Checkpoint:** the largest group is weekday evening with 5,447 fatal accidents.


### Analysis 3: Accident cause and vehicle involvement

**Write the entire cell.**

**Grain:** one row per accident cause and multi-vehicle flag.

**Required columns:** `accident_cause`, `multi_vehicle_flag`, `number_of_accidents`, `average_severity_score`.

Use named aggregation, round the average to two decimals, and sort by accident count descending, then average severity descending.

**Expected rows:** 167.


In [22]:
# TODO: Write the complete cause_vehicle_analysis chain.


display(cause_vehicle_analysis.head())
assert len(cause_vehicle_analysis) == 167

,accident_cause,multi_vehicle_flag,number_of_accidents,average_severity_score
55,driver's lack of attention to conveyance,True,71133,2.25
54,driver's lack of attention to conveyance,False,36655,1.79
80,incompatible velocity,False,31550,2.19
37,driver broke the laws of transit,True,25862,2.98
56,driver's lack of reaction,False,20816,2.06


**Checkpoint:** the first group is multi-vehicle accidents caused by driver's lack of attention to conveyance, with 71,133 accidents and an average severity score of 2.25.


### Analysis 4: Quarterly state hotspots

**Grain:** one row per state and quarter.

**Required columns:** `state_code`, `quarter`, `total_accidents`.

Group by the two grain columns, count with `size()`, rename the count, reset the index, and sort descending.

**Expected rows:** 108.


In [24]:
hotspot_groups = featured_accidents.groupby(
    ["state_code", "quarter"],
    dropna=False,
)

# TODO: Starting with hotspot_groups, create quarterly_hotspot_analysis.
display(quarterly_hotspot_analysis.head())
assert len(quarterly_hotspot_analysis) == 108

,state_code,quarter,total_accidents
40,MG,1,16266
42,MG,3,15201
43,MG,4,14995
41,MG,2,14859
92,SC,1,14477


**Checkpoint:** the first row is MG, quarter 1, with 16,266 accidents.


### Validate all four outputs

The analysis registry and grain columns are provided. Build one quality row per output with:

- table name
- row count
- column count
- duplicate count at the stated grain
- null count in the grain columns

Then assert that every output is nonempty and has no duplicate grain rows.


In [26]:
ANALYSES = {
    "SEVERITY_ANALYSIS": (
        severity_analysis,
        ["weather_condition", "road_classification"],
    ),
    "FATAL_TEMPORAL_PATTERNS": (
        fatal_temporal_patterns,
        ["time_of_day_category", "weekend_flag"],
    ),
    "CAUSE_VEHICLE_ANALYSIS": (
        cause_vehicle_analysis,
        ["accident_cause", "multi_vehicle_flag"],
    ),
    "QUARTERLY_HOTSPOT_ANALYSIS": (
        quarterly_hotspot_analysis,
        ["state_code", "quarter"],
    ),
}

# TODO: Build analysis_quality_summary with one dictionary per ANALYSES item.
# TODO: Display it and assert:
# row_count > 0
# duplicate_grain_rows == 0
# null_grain_values == 0


display(analysis_quality_summary)
assert (analysis_quality_summary["row_count"] > 0).all()
assert (analysis_quality_summary["duplicate_grain_rows"] == 0).all()
assert (analysis_quality_summary["null_grain_values"] == 0).all()


,table_name,row_count,column_count,duplicate_grain_rows,null_grain_values
0,SEVERITY_ANALYSIS,28,4,0,0
1,FATAL_TEMPORAL_PATTERNS,8,3,0,0
2,CAUSE_VEHICLE_ANALYSIS,167,4,0,0
3,QUARTERLY_HOTSPOT_ANALYSIS,108,3,0,0


### Analysis checkpoint

| Table | Rows | Columns | Duplicate grain rows | Null grain values |
|---|---:|---:|---:|---:|
| SEVERITY_ANALYSIS | 28 | 4 | 0 | 0 |
| FATAL_TEMPORAL_PATTERNS | 8 | 3 | 0 | 0 |
| CAUSE_VEHICLE_ANALYSIS | 167 | 4 | 0 | 0 |
| QUARTERLY_HOTSPOT_ANALYSIS | 108 | 3 | 0 | 0 |


## 8. Load Snowflake context from `snow.cfg`

This is your first time connecting to Snowflake from Python, so read each piece before running it.

- `ConfigParser` reads the `snow.cfg` text file, so your account and password never live in this notebook.
- The `[DEV]` section becomes a plain Python dictionary named `params`.
- `snowflake.connector.connect(**params)` opens the connection. `**params` spreads the dictionary into named arguments, so it becomes `connect(account=..., user=..., password=..., ...)`.

Your `student-work/week4/day4/snow.cfg` should look like this:

```ini
[DEV]
account = <your_snowflake_account_identifier>
user = <your_snowflake_username>
password = <your_snowflake_password>
role = DE
warehouse = COMPUTE_WH
database = TECHCATALYST
schema = <your_assigned_schema>
```

Replace `<your_snowflake_account_identifier>` with the account the instructor gives you, `<your_snowflake_username>` and `<your_snowflake_password>` with your own Snowflake login, and `<your_assigned_schema>` with your personal schema (`TECHCATALYST.<your_name>`).

### What is `authenticator`, and the browser fallback

The connector needs to know **how** to prove who you are. By default, when a `password` is present, it uses Snowflake's username-and-password sign-in (internally the `snowflake` authenticator). That is what the config above does, and what we use in class.

There is another option you should know about: adding the line `authenticator = externalbrowser`. That switches to **browser-based single sign-on**. Instead of reading a password, the connector opens a web browser so you sign in through an identity provider (Okta, AD FS, or another SAML provider), and no password is stored anywhere.

Use the browser option as a **fallback if your password sign-in fails** (for example, your account is configured for SSO). To switch, edit `snow.cfg`: remove the `password` line and add `authenticator = externalbrowser`.

```ini
[DEV]
account = <your_snowflake_account_identifier>
user = <your_snowflake_username>
authenticator = externalbrowser
role = DE
warehouse = COMPUTE_WH
database = TECHCATALYST
schema = <your_assigned_schema>
```

Nothing else in this notebook changes. `**params` simply passes `authenticator` to `connect` instead of `password`. The loader below accepts either sign-in style.

The repository-root `.gitignore` already ignores every `snow.cfg`, so your password (or browser session) stays on your machine. Keep secrets in `snow.cfg` only, never in this notebook, and do not create a nested `.gitignore`.

In [29]:
CONFIG_PATH = Path("snow.cfg")
CONFIG_SECTION = "DEV"

config = ConfigParser()
loaded_files = config.read(CONFIG_PATH)

assert loaded_files, (
    f"Could not read {CONFIG_PATH.resolve()}. "
    "Complete Activity 0 and keep snow.cfg beside this notebook."
)
assert CONFIG_SECTION in config, (
    f"Missing [{CONFIG_SECTION}] section in {CONFIG_PATH}."
)

params = {
    key: value.strip()
    for key, value in config[CONFIG_SECTION].items()
    if value.strip()
}

# These keys are always required, whichever sign-in style you use.
REQUIRED_CONNECTION_KEYS = {
    "account",
    "user",
    "role",
    "warehouse",
    "database",
    "schema",
}

missing_connection_keys = REQUIRED_CONNECTION_KEYS.difference(params)
assert not missing_connection_keys, (
    f"Missing connection keys: {sorted(missing_connection_keys)}"
)

# You must sign in with EITHER a password OR browser SSO.
uses_browser = params.get("authenticator") == "externalbrowser"
assert "password" in params or uses_browser, (
    "Add a password to snow.cfg, or set authenticator = externalbrowser "
    "to sign in with the browser."
)

placeholder_keys = {
    key
    for key, value in params.items()
    if value.startswith("<") or value.endswith(">")
}
assert not placeholder_keys, (
    f"Replace placeholders for: {sorted(placeholder_keys)}"
)

# Show the context WITHOUT the password, so nothing secret prints.
safe_context = {
    key: params[key]
    for key in ["user", "role", "warehouse", "database", "schema"]
    if key in params
}
if "authenticator" in params:
    safe_context["authenticator"] = params["authenticator"]
display(safe_context)

{'user': 'ATWAN',
 'role': 'DE',
 'warehouse': 'COMPUTE_WH',
 'database': 'TECHCATALYST',
 'schema': 'ATWAN'}

### Connect using dictionary unpacking

`**params` unpacks the dictionary so its keys become named arguments to `snowflake.connector.connect`.

For example, `{"account": "abc", "user": "sam"}` becomes `connect(account="abc", user="sam")`.

`connect` returns a **connection object**. You reuse it for every query in this notebook and for `write_pandas`, and you close it at the end with `conn.close()`. This connector is the core Snowflake Python SDK. Next week you will meet **Snowpark**, a higher-level Snowflake API that lets you work with DataFrames directly on Snowflake; it is built on top of the same connection ideas you are learning here.

Complete the one connection line, then run the context verification query.

In [ ]:
# TODO: Connect by unpacking params with **.
# conn =

print("Snowflake connection opened.")

# TODO: Run the provided context query after the connection succeeds.
CONTEXT_SQL = '''
SELECT
  CURRENT_USER(),
  CURRENT_ROLE(),
  CURRENT_WAREHOUSE(),
  CURRENT_DATABASE(),
  CURRENT_SCHEMA()
'''

# with conn.cursor() as cursor:
#     cursor.execute(CONTEXT_SQL)
#     connection_context = cursor.fetchone()

# print(connection_context)


Snowflake connection opened.
('ATWAN', 'DE', 'COMPUTE_WH', 'TECHCATALYST', 'ATWAN')


## 9. Create explicit transient target tables

You already created tables in SQL earlier today. Apply that DDL skill here.

The first target statement is completed as a model. Write the other three statements so their columns and types match the DataFrames exactly.

Use:

- `VARCHAR` for text
- `BOOLEAN` for flags
- `NUMBER` for counts and quarter
- `FLOAT` for average severity

Do not let the connector guess the target schema.


In [ ]:
TARGET_DDL = {
    "SEVERITY_ANALYSIS": """
        CREATE OR REPLACE TRANSIENT TABLE SEVERITY_ANALYSIS (
          WEATHER_CONDITION VARCHAR,
          ROAD_CLASSIFICATION VARCHAR,
          ACCIDENT_COUNT NUMBER,
          AVERAGE_SEVERITY_SCORE FLOAT
        )
    """,
    # TODO: Add FATAL_TEMPORAL_PATTERNS.
    # TODO: Add CAUSE_VEHICLE_ANALYSIS.
    # TODO: Add QUARTERLY_HOTSPOT_ANALYSIS.
}


In [ ]:
# TODO 1: Assert that TARGET_DDL and ANALYSES have the same table names.
# TODO 2: Open one cursor.
# TODO 3: Loop through TARGET_DDL and execute every statement.

# 1 given 
assert set(TARGET_DDL) == set(ANALYSES)

# 2 and 3 on you 


Created SEVERITY_ANALYSIS
Created FATAL_TEMPORAL_PATTERNS
Created CAUSE_VEHICLE_ANALYSIS
Created QUARTERLY_HOTSPOT_ANALYSIS


## 10. Write each DataFrame with `write_pandas`

`write_pandas` is a helper from `snowflake.connector.pandas_tools`. Behind one Python call it writes your DataFrame to local Parquet files, uploads them to a temporary Snowflake stage with `PUT`, and loads the table with `COPY INTO`. 

It returns a tuple of four values: `(success, num_chunks, num_rows, output)`.

- `success`: did the load succeed
- `num_chunks`: how many chunks it copied
- `num_rows`: how many rows it wrote
- `output`: the raw `COPY INTO` result

In the loop:

1. Copy the DataFrame.
2. Uppercase its columns so they match the unquoted Snowflake DDL.
3. Call `write_pandas`.
4. Pass `quote_identifiers=False` because the explicit target columns are unquoted uppercase identifiers. (Its default is `True`, which would wrap every name in double quotes and no longer match your table.)
5. Use `on_error="ABORT_STATEMENT"` so a bad row stops that statement instead of silently loading partial data.
6. Store success, chunk count, DataFrame rows, and written rows.

In [ ]:
snowflake_outputs = {
    table_name: frame.copy()
    for table_name, (frame, _) in ANALYSES.items()
}

write_results = []

for table_name, frame in snowflake_outputs.items():
    snowflake_frame = frame.copy()

    # TODO 1: Uppercase every column name.

    # TODO 2: Call write_pandas with the required arguments.
    # It returns: success, chunk_count, written_rows, copy_output.

    # TODO 3: Append one evidence dictionary to write_results.
    pass

# TODO 4: Convert write_results to write_results_df and display it.


write_results_df = pd.DataFrame(write_results)
display(write_results_df.drop(columns="copy_output"))

/var/folders/48/j6k669vx63qd_68k2_502cl40000gn/T/ipykernel_58600/593232921.py:15: UserWarning: Pandas Dataframe has non-standard index of type <class 'pandas.Index'> which will not be written. Consider changing the index to pd.RangeIndex(start=0,...,step=1) or call reset_index() to keep index as column(s)
  success, chunk_count, written_rows, copy_output = write_pandas(
/var/folders/48/j6k669vx63qd_68k2_502cl40000gn/T/ipykernel_58600/593232921.py:15: UserWarning: Pandas Dataframe has non-standard index of type <class 'pandas.Index'> which will not be written. Consider changing the index to pd.RangeIndex(start=0,...,step=1) or call reset_index() to keep index as column(s)
  success, chunk_count, written_rows, copy_output = write_pandas(
/var/folders/48/j6k669vx63qd_68k2_502cl40000gn/T/ipykernel_58600/593232921.py:15: UserWarning: Pandas Dataframe has non-standard index of type <class 'pandas.Index'> which will not be written. Consider changing the index to pd.RangeIndex(start=0,...,step

,table_name,success,chunk_count,dataframe_rows,written_rows
0,SEVERITY_ANALYSIS,True,1,28,28
1,FATAL_TEMPORAL_PATTERNS,True,1,8,8
2,CAUSE_VEHICLE_ANALYSIS,True,1,167,167
3,QUARTERLY_HOTSPOT_ANALYSIS,True,1,108,108


## 11. Reconcile Snowflake row counts

For each table, compare:

- pandas DataFrame rows
- rows reported by `write_pandas`
- `SELECT COUNT(*)` from Snowflake

A pipeline is not complete until all three counts match.


In [ ]:
# TODO: Build written_row_lookup and success_lookup from write_results.
# Given below
written_row_lookup = {
    record["table_name"]: record["written_rows"]
    for record in write_results
}
success_lookup = {
    record["table_name"]: record["success"]
    for record in write_results
}


# TODO: Query COUNT(*) for every table in snowflake_outputs.
# TODO: Build reconciliation with:
# table_name, dataframe_rows, written_rows, snowflake_rows, counts_match.



# TODO: Display it and assert that every write and count comparison passed.
# Given below
reconciliation = pd.DataFrame(reconciliation_records)
display(reconciliation)

assert all(success_lookup.values())
assert reconciliation["counts_match"].all()

,table_name,dataframe_rows,written_rows,snowflake_rows,counts_match
0,SEVERITY_ANALYSIS,28,28,28,True
1,FATAL_TEMPORAL_PATTERNS,8,8,8,True
2,CAUSE_VEHICLE_ANALYSIS,167,167,167,True
3,QUARTERLY_HOTSPOT_ANALYSIS,108,108,108,True


### Expected reconciliation

| Table | DataFrame rows | Written rows | Snowflake rows | Match |
|---|---:|---:|---:|---|
| SEVERITY_ANALYSIS | 28 | 28 | 28 | True |
| FATAL_TEMPORAL_PATTERNS | 8 | 8 | 8 | True |
| CAUSE_VEHICLE_ANALYSIS | 167 | 167 | 167 | True |
| QUARTERLY_HOTSPOT_ANALYSIS | 108 | 108 | 108 | True |


In [38]:
conn.close()
print("Snowflake connection closed.")


Snowflake connection closed.


## 12. Reflection

Add a Markdown cell and answer:

1. Which work happened in pandas, and which work happened in Snowflake?
2. Why did you build the rename dictionary instead of receiving a completed mapping?
3. Why was the IQR implementation provided, and what interpretation work remained yours?
4. Why did the pipeline pre-create transient tables instead of letting a library guess the schema?
5. What does `**params` do in the connection call?
6. How does `write_pandas` connect to the temporary stage and `COPY INTO` concepts?
7. Which analysis would you rebuild as a Snowflake view, and why?
